In [37]:
import polars as pl

year = 2022
poll_data = pl.scan_parquet(
    f"s3://arthurmanceau/election_modeling_uhcp/data/polls/presidentiel/{year}/polls_t1.parquet",
    storage_options={
        "aws_endpoint_url": "https://minio.lab.sspcloud.fr",
        "aws_region": "us-east-1",
    },
    credential_provider=pl.CredentialProviderAWS(
        profile_name="default",
        region_name="us-east-1",
    ),
).collect()

results = pl.scan_parquet(
    f"s3://arthurmanceau/election_modeling_uhcp/data/output/results/results_synth_{year}_pres_['TD', 'TG', 'par']_0.5.0.parquet",
    storage_options={
        "aws_endpoint_url": "https://minio.lab.sspcloud.fr",
        "aws_region": "us-east-1",
    },
    credential_provider=pl.CredentialProviderAWS(
        profile_name="default",
        region_name="us-east-1",
    ),
    glob=False,
).collect()

In [38]:
td_true = (
    results.filter(pl.col("index") == "pvoteTD").get_column(f"{year}_pres_true").item(0)
)
tg_true = (
    results.filter(pl.col("index") == "pvoteTG").get_column(f"{year}_pres_true").item(0)
)

In [39]:
poll_data.with_columns(
    mae_poll_tg=pl.col("TG") - tg_true,
    mae_poll_td=pl.col("TD") - td_true,
).select("D", "G", "C", "CG", "CD").sum_horizontal()

sum
f64
110.528075
111.678075
111.678075
111.178075
110.678075
…
100.678075
98.178075
101.678075
